In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'd:\\projects\\project-13 frad detection\\fraud detection\\fraud-Detection-elliptic-bitcoin'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    data_path: Path
    params_epochs: int
    params_lr: float

In [ ]:
from src.frauddetection.constants import *
from src.frauddetection.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.model_training
        prepare_base_model = self.config.prepare_base_model
        transformation = self.config.data_transformation
        
        create_directories([training.root_dir])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.base_model_path),
            data_path=Path(transformation.root_dir),
            params_epochs=self.params.EPOCHS,
            params_lr=self.params.LEARNING_RATE
        )

        return training_config

In [ ]:
import torch
import torch.nn as nn
from src.frauddetection import logger
import torch_geometric
import torch_geometric.data.data
import torch_geometric.data.storage


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self, model_class):
        # Initialize architecture and load weights
        self.model = model_class(
            in_channels=165, # Standard for Elliptic
            hidden_channels=64,
            out_channels=2
        )
        self.model.load_state_dict(torch.load(self.config.updated_base_model_path))
        self.model.train()

    def train(self):
        # Load the transformed graph data
        
        # Register safe globals before loading the .pt file
        torch.serialization.add_safe_globals([
            torch_geometric.data.data.DataTensorAttr,
            torch_geometric.data.data.DataEdgeAttr,
            torch_geometric.data.storage.GlobalStorage
        ])
        data = torch.load(os.path.join(self.config.data_path, "transformed_data.pt"))
        
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.config.params_lr)
        criterion = nn.CrossEntropyLoss()

        logger.info(f"Starting training for {self.config.params_epochs} epochs...")

        for epoch in range(self.config.params_epochs):
            optimizer.zero_grad()
            out = self.model(data.x, data.edge_index)
            
            # Train only on nodes within the training mask
            loss = criterion(out[data.train_mask], data.y_binary[data.train_mask])
            loss.backward()
            optimizer.step()

            if epoch % 10 == 0:
                logger.info(f"Epoch {epoch}: Loss {loss.item():.4f}")

        self.save_model(self.model, self.config.trained_model_path)

    @staticmethod
    def save_model(model: torch.nn.Module, path: Path):
        torch.save(model.state_dict(), path)
        logger.info(f"Trained model saved at: {path}")

In [ ]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    
    # Passing the class definition from the previous stage
    training.get_base_model(FraudSAGE) 
    training.train()
except Exception as e:
    raise e